# 1. Classification Example: Iris Dataset

In this notebook chapter, we’ll walk through a classical classification task using Scikit‑Learn and the famous **Iris dataset**. You’ll see how to build a clean, modular pipeline and evaluate your model with standard metrics.

## Objective  
Predict the species of an iris flower (`setosa`, `versicolor`, `virginica`) based on morphological measurements.

## Dataset Overview  
- **Samples:** 150  
- **Features (numeric):**  
  - Sepal length (cm)  
  - Sepal width (cm)  
  - Petal length (cm)  
  - Petal width (cm)  
- **Classes:** 3 (50 samples each)

## Workflow  
1. **Load Data** (`sklearn.datasets.load_iris`)  
2. **Exploratory Data Analysis**  
   - Feature distributions  
   - Pairwise scatter plots  
3. **Train/Test Split**  
   - 70% training / 20% test  / 10% validation  
   - Stratified sampling to preserve class proportions  
4. **Preprocessing & Pipeline**  
   - `StandardScaler` for feature normalization  
   - `LogisticRegression` as a baseline classifier  
5. **Model Training**  
6. **Evaluation**  
   - **Accuracy**  
   - **Confusion Matrix**  
   - **Precision, Recall, F1‑Score** (`classification_report`)

## Extension Ideas  
- Swap `LogisticRegression` for `RandomForestClassifier` or `SVC`  
- Add `PCA` for dimensionality reduction  
- Perform hyperparameter tuning with `GridSearchCV` or `RandomizedSearchCV`  
- Visualize learning curves and feature importances  

By the end of this chapter, you will have a reproducible classification pipeline and a solid understanding of key evaluation metrics.  

## 1.1 Imports & Data Loading  
First, we import everything we need and load the Iris dataset into a pandas DataFrame for easy EDA.


In [ ]:
# Cell 1: Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)
from sklearn.model_selection import GridSearchCV

# Ensure plots appear inline
%matplotlib inline

In [ ]:
# Cell 2: Load Iris into DataFrame
iris = load_iris()
X = iris.data
y = iris.target
df = pd.DataFrame(X, columns=iris.feature_names)
df['target'] = y
df['species'] = df['target'].map(dict(enumerate(iris.target_names)))

df.head()

In [ ]:
## 2.1 Inspecting the Iris Dataset & Sample Visualization

# After loading the data, let’s take a closer look at the `iris` object itself, inspect its contents, and then visualize one example flower of each species (in feature‐space) to get a feel for what the measurements represent.

# Cell 3: Inspect the iris Bunch
print("Keys of the iris dataset:", iris.keys(), "\n")
print("Feature names:", iris.feature_names)
print("Target names:", iris.target_names, "\n")

# Show a brief description
print(iris.DESCR.split("\n")[0:10])  # first 10 lines of the description

In [ ]:
# Cell 4: Bar‐plot one sample per species

# Map species index to name
species_names = iris.target_names
feature_names = iris.feature_names

# pick the first index of each species
indices = [np.where(y == i)[0][0] for i in range(len(species_names))]

fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
for ax, idx, name in zip(axes, indices, species_names):
    values = X[idx]
    ax.bar(np.arange(len(feature_names)), values)
    ax.set_xticks(np.arange(len(feature_names)))
    ax.set_xticklabels(feature_names, rotation=45, ha='right')
    ax.set_title(f"{name.capitalize()} (sample {idx})")
    ax.set_ylabel("cm")

plt.tight_layout()
plt.suptitle("Feature values for one sample of each Iris species", y=1.02)
plt.show()

In [ ]:
# Cell 4: Train/Validation/Final‑Test Split (70% / 20% / 10%)

# 1) Hold out 10% as final test set
X_rest, X_test, y_rest, y_test = train_test_split(
    X, y,
    test_size=0.10,        # 10% of original
    random_state=69,
    stratify=y
)

# 2) From the remaining 90%, hold out 20% of the ORIGINAL as validation:
val_ratio = 0.20 / 0.90   # = 0.2222...
X_train, X_val, y_train, y_val = train_test_split(
    X_rest, y_rest,
    test_size=val_ratio,
    random_state=69,
    stratify=y_rest
)

print(f"Train samples:      {X_train.shape[0]}  ({X_train.shape[0]/len(X):.0%})")
print(f"Validation samples: {X_val.shape[0]}  ({X_val.shape[0]/len(X):.0%})")
print(f"Final‑Test samples: {X_test.shape[0]}  ({X_test.shape[0]/len(X):.0%})")

In [ ]:
## Preprocessing & Pipeline

# Now that we have our splits, let’s build a reusable pipeline that will:
# 1. **Scale** all features to mean 0 and variance 1  
# 2. **Fit** a baseline `LogisticRegression` classifier  

# Cell 5: Build the pipeline


pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(random_state=69))
])

# Train on the 70% training set
pipeline.fit(X_train, y_train)
print("Pipeline training complete.")

In [ ]:
# 1) Generate synthetic binary data with a known sigmoid “truth”
np.random.seed(0)
X = np.linspace(-10, 10, 200)
p_true = 1 / (1 + np.exp(-(0.8 * (X - 2))))  # true underlying probability
y = np.random.binomial(1, p_true)            # sample 0/1 labels

# 2) Fit Logistic Regression on the raw data
model = LogisticRegression()
model.fit(X.reshape(-1, 1), y)

# 3) Compute model‐predicted probabilities on a fine grid
xx = np.linspace(-10, 10, 400)
p_pred = model.predict_proba(xx.reshape(-1, 1))[:, 1]

# 4) Estimate observed fraction of positives in bins
bins = np.linspace(-10, 10, 11)
bin_indices = np.digitize(X, bins)
bin_centers = (bins[:-1] + bins[1:]) / 2
bin_frac = [y[bin_indices == i].mean() if np.any(bin_indices == i) else np.nan
            for i in range(1, len(bins))]

# 5) Plot
plt.figure(figsize=(6, 4))
# scatter of observed positive fraction per bin
plt.scatter(bin_centers, bin_frac, label='Observed fraction (per bin)', s=50, edgecolors='k')
# true underlying curve
plt.plot(xx, 1/(1 + np.exp(-(0.8*(xx - 2)))), linestyle='--', label='True probability')
# fitted logistic curve
plt.plot(xx, p_pred, linewidth=2, label='Fitted sigmoid (LR)')
plt.xlabel('Feature value (X)')
plt.ylabel('Probability of y=1')
plt.title('Logistic Regression Approximates Bin‐Averaged Data')
plt.legend()
plt.grid(True)
plt.show()

**Why use LogisticRegression for the Iris example?**

- **Classification, not regression**  
  We assign each flower to one of three species, not predict a continuous value.

- **Probability estimates via log‑odds**  
  LogisticRegression models class probabilities using Softmax (multiclass) or Sigmoid (binary):

  $$
  P(y = k \mid \mathbf{x}) \;=\;
  \frac{\exp\bigl(\mathbf{w}_k^\top \mathbf{x} + b_k\bigr)}
       {\sum_j \exp\bigl(\mathbf{w}_j^\top \mathbf{x} + b_j\bigr)}
  $$

  $$
  \sigma(z) \;=\; \frac{1}{1 + e^{-z}}, 
  \quad
  P(y=1 \mid \mathbf{x}) = \sigma\bigl(\mathbf{w}^\top \mathbf{x} + b\bigr)
  $$

- **Linear decision boundary**  
  The decision surface is the hyperplane:

  $$
  \mathbf{w}^\top \mathbf{x} + b = 0
  $$

  In 1D this yields a threshold \(x_0\) where 
  $$
  (\,w\,x_0 + b = 0 \implies x_0 = -\,b/w)
  $$.

- **Multiclass support**  
  - **Multinomial (Softmax):** optimizes all classes jointly  
  - **One‑vs‑Rest:** trains a binary logistic model per class

- **Interpretability & efficiency**  
  Each weight $(w_{k,i})$ indicates how feature \(i\) affects the log‑odds of class \(k\). Training scales efficiently on high‑dimensional tabular data.

> **In short:** LogisticRegression builds a linear decision surface $(\mathbf{w}^\top \mathbf{x} + b = 0)$ in multi‑dimensional space, combining all features to optimally separate classes.

In [ ]:
## 1.5 Validation Evaluation

# Assess the pipeline on the 20% validation set. We’ll compute accuracy, display a confusion matrix, and print precision/recall/F1 scores.

# Cell 6: Validation set predictions and evaluation

# Predict on validation set
y_val_pred = pipeline.predict(X_val)

# 1) Accuracy
val_acc = accuracy_score(y_val, y_val_pred)
print(f"Validation Accuracy: {val_acc:.2f}")

# 2) Confusion Matrix
cm_val = confusion_matrix(y_val, y_val_pred)
disp_val = ConfusionMatrixDisplay(confusion_matrix=cm_val, display_labels=iris.target_names)
disp_val.plot(xticks_rotation='vertical')
plt.title("Validation Confusion Matrix")
plt.show()

# 3) Classification Report
print("Validation Classification Report:\n")
print(classification_report(y_val, y_val_pred, target_names=iris.target_names))

In [ ]:
## 1.6 Final‑Test Evaluation

# Once you’ve settled on your pipeline (and any hyperparameter tuning on the validation set), evaluate the *final* 10% hold‑out set for an unbiased performance estimate:

# Cell 7: Final‑test set evaluation

# Predict on final‑test set
y_test_pred = pipeline.predict(X_test)

# 1) Accuracy
test_acc = accuracy_score(y_test, y_test_pred)
print(f"Final‑Test Accuracy: {test_acc:.2f}")

# 2) Confusion Matrix
cm_test = confusion_matrix(y_test, y_test_pred)
disp_test = ConfusionMatrixDisplay(confusion_matrix=cm_test, display_labels=iris.target_names)
disp_test.plot(xticks_rotation='vertical')
plt.title("Final‑Test Confusion Matrix")
plt.show()

# 3) Classification Report
print("Final‑Test Classification Report:\n")
print(classification_report(y_test, y_test_pred, target_names=iris.target_names))

In [ ]:
# Cell 8: Hyperparameter tuning


param_grid = {
    'clf__C': [0.01, 0.1, 1, 10, 100],
    'clf__penalty': ['l2']
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print(f"Best cross‑val score: {grid.best_score_:.2f}")

# Evaluate tuned model on validation or final‑test set as before